In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import HTML, Markdown, display

from ne_syntax_consistency.spans import LABELS, sample_examples, top_patterns
from ne_syntax_consistency.syntax_vis import Style
from ne_syntax_consistency.viz import record_svg

STYLE = Style()


def _project_root() -> Path:
    # Jupyter käivitab kerneli notebooki kaustas; otsime üles pyproject.toml-i
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("pyproject.toml not found")


DATA = _project_root() / "data"
RESULTS = _project_root() / "results" / "span_trees"

In [ ]:
SPAN_FILL, SPAN_ARC, CONTEXT_ARC, NE_FILL = (
    STYLE.highlight_fill,
    STYLE.highlight_arc,
    STYLE.context_arc,
    STYLE.box_fill,
)
display(
    Markdown(f"""
# NE spannide sisemised puud

Iga NE span annab täpselt ühe mustri: tema sõnade ja nendevaheliste kaarte puu
(`scripts/span_trees.py`, `ne_syntax_consistency.spans`). Mustrid ja tugi loetakse
skripti salvestatud tulemusfailidest; lausetest laaditakse ainult näited. Mustri tugi = spannide arv,
mille terve sisemine puu on just see muster; alammustreid ei loeta. Uurime mustreid,
mille tugi on vähemalt piirväärtus (vaikimisi 10), ja iga mustri kohta juhuslikke näitelauseid.

Joonisel: <span style="background:{SPAN_FILL};padding:0 4px">span</span>,
<b style="color:{SPAN_ARC}">paks kaar</b> = spanni sisene kaar (muster),
<b style="color:{CONTEXT_ARC}">katkendkaar</b> = spanni juure seos välise ülemaga.
Teised NE-d on <span style="background:{NE_FILL};padding:0 4px">helesinisel taustal</span>, tüüp kasti all.
Sõna all on sõnaliik ja kääne (`morph_analysis`); "…" = peidetud sõnad.
""")
)

In [ ]:
# Parameetrid. Tulemusfailid tehakse skriptiga, nt:
#   uv run python scripts/span_trees.py data/full --label type \
#       --max-id 19256634 --name full_nochat
LABEL = "type"  # sõnade märgend mustris, üks LABELS-ist
NAME = (
    "full_nochat"  # tulemuse nimi: full_nochat, full_chat, full, sample_200000_s1, ...
)
SENTENCES = "full"  # lausete Parquet-fail või kaust data/ all (näidete jaoks)
MIN_SUPPORT = 10  # min tugi
TOP_N = 10  # mustreid tüübi kohta
N_EXAMPLES = 4  # näiteid mustri kohta (sama valik kui scripts/export_examples.py)
WINDOW = 5  # sõnu iga spanni sõna ümber (0 = ainult span ja ülemad, None = terve lause)
DISTINCT = True  # iga spanni tekst mustri kohta ülimalt korra
SEED = 1

assert LABEL in LABELS, list(LABELS)
STEM = RESULTS / f"span_trees_{LABEL}_{NAME}"
print("Saadaval:", sorted(p.name for p in RESULTS.glob("span_trees_*_spans.parquet")))

In [ ]:
table = pd.concat(
    [pd.read_csv(f"{STEM}.csv"), pd.read_csv(f"{STEM}_rare.csv")], ignore_index=True
).sort_values(["nertag", "support"], ascending=[True, False])
# üks rida spanni kohta; mustrid ja tüübid sõnastikuna, et kogu korpus mahuks mällu
spans = pq.read_table(
    f"{STEM}_spans.parquet",
    columns=["text_id", "ne_index", "nertag", "pattern", "ne_text"],
    read_dictionary=["nertag", "pattern"],
).to_pandas()
len(table), len(spans)

In [ ]:
# Iga tüübi top-N mustrit (tugi >= min tugi) ja juhuslikud näited iga mustri kohta
top = top_patterns(table, MIN_SUPPORT, TOP_N)
examples = sample_examples(spans, top, N_EXAMPLES, SEED, DISTINCT)
sentences = {
    s["text_id"]: s
    for s in pq.read_table(
        DATA / SENTENCES,
        columns=[
            "text_id",
            "words",
            "pos",
            "form",
            "head",
            "deprel",
            "ne_tag",
            "ne_words",
        ],
        filters=[("text_id", "in", sorted(set(examples["text_id"].tolist())))],
    ).to_pylist()
}
len(examples), len(sentences)

In [ ]:
_summary = (
    table.assign(frequent=table["support"] >= MIN_SUPPORT)
    .groupby(["nertag", "frequent"])
    .agg(patterns=("pattern", "size"), spans=("support", "sum"))
    .unstack("frequent", fill_value=0)
)
display(Markdown(f"### Mustrite arv tüübi kaupa (tugi ≥ {MIN_SUPPORT}: True)"))
display(_summary)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(top[["nertag", "support", "share_of_type", "n_roots", "pattern"]])

In [ ]:
def show_examples(nertag):
    """Joonistab tüübi top-mustrid koos juhuslike näitelausetega."""
    display(Markdown(f"## {nertag}"))
    for p in top[top["nertag"] == nertag].itertuples():
        display(
            Markdown(
                f"#### `{p.pattern}`\n\ntugi **{p.support}** "
                f"({p.share_of_type:.1%} tüübi spannidest), juuri {p.n_roots}"
            )
        )
        _ex = examples[
            (examples["nertag"] == p.nertag) & (examples["pattern"] == p.pattern)
        ]
        for row in _ex.itertuples():
            s = sentences[row.text_id]
            svg = record_svg(s, s["ne_words"][row.ne_index], WINDOW, STYLE)
            display(
                Markdown(
                    f"`{row.text_id}` · ({row.ne_text_count}× mustris) "
                    f"{' '.join(s['words'])}"
                )
            )
            display(HTML(f'<div style="overflow-x:auto">{svg}</div>'))


sorted(top["nertag"].unique())

In [ ]:
show_examples("ORG")

In [ ]:
show_examples("PER")

In [ ]:
show_examples("LOC")